# Evidently AI — Data Drift Detection on Diabetes Dataset

This notebook demonstrates how to use [Evidently AI](https://www.evidentlyai.com/) to detect **data drift** between a reference dataset and a simulated production dataset.

## What Changed from the Original
The original lab used the **Adult dataset** (from OpenML), which mixes numerical and categorical features. This version uses the **Diabetes dataset** from `sklearn`, which is fully numerical and requires no external download.

## Dataset
- **Source**: `sklearn.datasets.load_diabetes`
- **Samples**: 442
- **Features**: 10 numerical (age, sex, bmi, bp, s1–s6)
- **Target**: disease progression score (continuous)

## Drift Simulation
We take the first 60% of the data as the **reference** (training-time distribution) and introduce a shift in the production slice by scaling `bmi` and `bp` — two clinically meaningful features — to simulate population drift.

In [ ]:
# Install dependencies if needed
# !pip install evidently scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes

from evidently import Dataset, DataDefinition
from evidently.presets import DataDriftPreset
from evidently import Report

## 1. Load the Diabetes Dataset

In [ ]:
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Split into Reference and Production

In [ ]:
split_idx = int(len(df) * 0.6)
reference_df = df.iloc[:split_idx].reset_index(drop=True)
production_df = df.iloc[split_idx:].reset_index(drop=True)

print(f"Reference samples : {len(reference_df)}")
print(f"Production samples: {len(production_df)}")

## 3. Simulate Production Drift

We shift the `bmi` and `bp` columns in the production slice to simulate a realistic population shift (e.g. patients with higher BMI entering the system over time).

In [ ]:
production_drifted = production_df.copy()
production_drifted['bmi'] = production_drifted['bmi'] * 1.3
production_drifted['bp']  = production_drifted['bp']  + 0.05

print("Reference bmi mean :", reference_df['bmi'].mean().round(4))
print("Production bmi mean:", production_drifted['bmi'].mean().round(4))

## 4. Define the Data Schema

In [ ]:
numerical_columns = list(diabetes.feature_names) + ['target']

schema = DataDefinition(
    numerical_columns=numerical_columns
)

print("Numerical columns:", numerical_columns)

## 5. Create Evidently Datasets

In [ ]:
reference_dataset  = Dataset.from_pandas(reference_df,     data_definition=schema)
production_dataset = Dataset.from_pandas(production_drifted, data_definition=schema)

## 6. Generate the Data Drift Report

In [ ]:
report = Report(metrics=[DataDriftPreset()])
my_eval = report.run(
    reference_data=reference_dataset,
    current_data=production_dataset
)
my_eval

## 7. Save Report Locally

In [ ]:
my_eval.save_html("diabetes_drift_report.html")
print("Report saved to diabetes_drift_report.html")

---
## Optional: Upload to Evidently Cloud

If you have an Evidently Cloud account, you can push the report for persistent dashboards:

```python
from evidently.ui.workspace.cloud import CloudWorkspace

ws = CloudWorkspace(token="YOUR_TOKEN", url="https://app.evidently.cloud")
project = ws.create_project("Diabetes Drift Monitoring")
ws.add_run(project.id, my_eval, include_data=False)
```